# OMNIS-COURT LLM Server v7.9 FINAL
## Qwen3-8B + 128K Context (4-bit weights + 8-bit KV cache)

### Instructions:
1. Runtime -> Factory reset runtime
2. Runtime -> Change runtime type -> T4 GPU
3. Run All (Ctrl+F9)
4. Wait 8-12 minutes for model loading
5. Copy LLM + Jina URLs from Cell 7
6. Paste into config/platforms.json
7. Close tab (anti-idle is active)

### Architecture:
- Model: Qwen3-8B (4-bit quantization, ~5GB)
- Context: 128K tokens (RoPE scaling 4x)
- KV Cache: 8-bit quantization (~9.4GB)
- Server: FastAPI + OpenAI-compatible
- Total VRAM: ~15GB / 16GB T4

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES (Stable versions)
# ============================================================
import os
import sys
import subprocess

# Disable telemetry
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('Installing dependencies...')

# Core ML packages (compatible versions)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch>=2.1.0',
    'transformers>=4.45.0',
    'accelerate>=0.34.0',
    'bitsandbytes>=0.44.0',
    'quanto>=0.2.0',  # For 8-bit KV cache
    'safetensors>=0.4.0',
    'sentencepiece>=0.2.0'
], check=True)

# Server packages
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi>=0.115.0',
    'uvicorn[standard]>=0.32.0',
    'pydantic>=2.9.0',
    'nest-asyncio>=1.6.0',
    'requests>=2.32.0'
], check=True)

# Content extraction
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'trafilatura>=1.12.0'
], check=True)

# Install cloudflared for tunnels
print('Installing cloudflared...')
subprocess.run([
    'curl', '-sL',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '-o', '/usr/local/bin/cloudflared'
], check=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

# Verify installations
print('\nVerifying installations...')
packages = [
    ('torch', 'torch'),
    ('transformers', 'transformers'),
    ('bitsandbytes', 'bitsandbytes'),
    ('accelerate', 'accelerate'),
    ('quanto', 'quanto'),
    ('fastapi', 'fastapi'),
    ('uvicorn', 'uvicorn'),
    ('trafilatura', 'trafilatura'),
    ('requests', 'requests')
]

all_ok = True
for name, module in packages:
    try:
        __import__(module)
        print(f'  [OK] {name}')
    except Exception as e:
        print(f'  [FAIL] {name}: {e}')
        all_ok = False

# Verify cloudflared
cf_result = subprocess.run(['cloudflared', '--version'],
                           capture_output=True, text=True)
print(f'  [OK] cloudflared: {cf_result.stdout.strip()}')

if all_ok:
    print('\n=== ALL DEPENDENCIES READY ===')
    print('Next: Restart runtime, then Run All again')
else:
    print('\n=== SOME PACKAGES FAILED ===')
    print('Please check errors above')

In [ ]:
# ============================================================
# CELL 2: ENVIRONMENT CHECK
# ============================================================
import torch
import sys
import subprocess

print('=== ENVIRONMENT CHECK ===')
print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'CUDA version: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Memory: {total_mem:.1f} GB')
    print(f'Compute capability: {torch.cuda.get_device_capability(0)}')
else:
    print('[ERROR] No GPU detected! Please enable T4 GPU in Runtime settings.')
    raise RuntimeError('GPU required')

# Check transformers version
import transformers
print(f'Transformers: {transformers.__version__}')

# Check bitsandbytes
import bitsandbytes
print(f'bitsandbytes: {bitsandbytes.__version__}')

# Check quanto
import quanto
print(f'quanto: {quanto.__version__}')

print('\n=== ENVIRONMENT CHECK PASSED ===')

In [ ]:
# ============================================================
# CELL 3: ANTI-IDLE
# ============================================================
from IPython.display import display, Javascript

js_code = '''
setInterval(function() {
    var btn = document.querySelector('colab-runbutton');
    if (!btn) {
        btn = document.querySelector('[aria-label="Run cell"]');
    }
    if (btn) {
        btn.click();
        console.log('Anti-idle: clicked at ' + new Date().toISOString());
    }
}, 240000);
console.log('Anti-idle active (4 minute interval)');
'''

display(Javascript(js_code))
print('Anti-idle active (4-minute interval)')
print('Safe to close tab after all cells complete')

In [ ]:
# ============================================================
# CELL 4: LOAD QWEN3-8B (4-bit weights + 8-bit KV cache)
# ============================================================
import torch
import time
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoConfig,
    BitsAndBytesConfig
)

print('=== LOADING QWEN3-8B ===')
print('Configuration:')
print('  - Weights: 4-bit (bitsandbytes NF4)')
print('  - KV Cache: 8-bit (quanto)')
print('  - Context: 128K tokens (RoPE scaling 4x)')
print('  - Expected VRAM: ~15GB')
print('  - Expected load time: 5-8 minutes')
print()

MODEL_ID = 'Qwen/Qwen3-8B'
MAX_CONTEXT_LEN = 131072  # 128K tokens

# Configure 4-bit quantization for weights
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=['lm_head']
)

# Load tokenizer first
print('[1/4] Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side='left',
    truncation_side='left'
)

# Set max length for tokenizer
tokenizer.model_max_length = MAX_CONTEXT_LEN

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f'  Tokenizer loaded (vocab: {tokenizer.vocab_size})')

# Load config and apply RoPE scaling
print('[2/4] Loading config with RoPE scaling...')
config = AutoConfig.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

# Apply RoPE scaling for 128K context (32K * 4 = 128K)
if hasattr(config, 'rope_scaling'):
    config.rope_scaling = {
        'type': 'dynamic',
        'factor': 4.0
    }
    config.max_position_embeddings = MAX_CONTEXT_LEN
    print(f'  RoPE scaling applied: 32K -> 128K')
else:
    print('  Warning: Model does not support rope_scaling')

# Load model with 4-bit weights
print('[3/4] Loading model (this takes 5-8 minutes)...')
start_time = time.time()

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        config=config,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        # Use static cache for better memory management
        attn_implementation='eager',  # safer than flash_attention_2
    )
except Exception as e:
    print(f'  [ERROR] Failed to load model: {e}')
    print('  Falling back to eager loading without device_map...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        config=config,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True
    )

load_time = time.time() - start_time
print(f'  Model loaded in {load_time:.1f} seconds')

# Set model to eval mode
model.eval()

# Disable thinking mode by default (for Qwen3)
if hasattr(model, 'generation_config'):
    model.generation_config.temperature = 0.7
    model.generation_config.top_p = 0.9
    model.generation_config.do_sample = True

# Check VRAM usage
print('[4/4] Checking VRAM usage...')
if torch.cuda.is_available():
    torch.cuda.synchronize()
    mem_used = torch.cuda.memory_allocated() / (1024**3)
    mem_reserved = torch.cuda.memory_reserved() / (1024**3)
    mem_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    
    print(f'  Allocated: {mem_used:.2f} GB')
    print(f'  Reserved: {mem_reserved:.2f} GB')
    print(f'  Total: {mem_total:.2f} GB')
    print(f'  Free for KV cache: {mem_total - mem_reserved:.2f} GB')
    
    if mem_reserved > 14.5:
        print('  [WARNING] High memory usage, KV cache may be limited')
    else:
        print('  [OK] Memory within budget')

# Store globals for server
GLOBAL_MODEL = model
GLOBAL_TOKENIZER = tokenizer
GLOBAL_MAX_CONTEXT = MAX_CONTEXT_LEN

print()
print('=== MODEL READY ===')
print(f'Model: {MODEL_ID}')
print(f'Context window: {MAX_CONTEXT_LEN} tokens')
print(f'Pad token: {tokenizer.pad_token}')

In [ ]:
# ============================================================
# CELL 5: START FASTAPI SERVER (OpenAI-compatible)
# ============================================================
import torch
import time
import threading
import requests
import json
import hashlib
import traceback
from datetime import datetime
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from typing import List, Optional, Dict, Any
import uvicorn
import nest_asyncio

# Apply nest_asyncio to allow nested event loops in Colab
nest_asyncio.apply()

print('=== STARTING FASTAPI SERVER ===')

# Define request/response models
class Message(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str = 'qwen3'
    messages: List[Message]
    max_tokens: int = 4096
    temperature: float = 0.7
    top_p: float = 0.9
    stream: bool = False

# Create FastAPI app
app = FastAPI(
    title='OMNIS-COURT LLM Server',
    version='7.9'
)

# Stats tracking
stats = {
    'total_requests': 0,
    'total_tokens': 0,
    'errors': 0,
    'start_time': datetime.now()
}

@app.get('/health')
async def health():
    """Health check endpoint"""
    return {
        'status': 'ok',
        'model': 'qwen3-8b',
        'uptime_seconds': (datetime.now() - stats['start_time']).total_seconds(),
        'requests_served': stats['total_requests']
    }

@app.get('/v1/models')
async def list_models():
    """List available models (OpenAI-compatible)"""
    return {
        'data': [{
            'id': 'qwen3',
            'object': 'model',
            'created': int(time.time()),
            'owned_by': 'omnis-court'
        }]
    }

@app.post('/v1/chat/completions')
async def chat_completions(request: ChatRequest):
    """Chat completions endpoint (OpenAI-compatible)"""
    stats['total_requests'] += 1
    
    try:
        # Convert messages to dict format
        messages = [{'role': m.role, 'content': m.content} for m in request.messages]
        
        # Handle Qwen3 thinking mode (disable for speed)
        # Add /no_think to system message if present
        processed_messages = []
        for msg in messages:
            if msg['role'] == 'user' and not any(
                m['role'] == 'system' for m in messages
            ):
                # If no system message, prepend /no_think to user message
                if not msg['content'].startswith('/no_think'):
                    processed_messages.append({
                        'role': 'user',
                        'content': '/no_think\n' + msg['content']
                    })
                else:
                    processed_messages.append(msg)
            else:
                processed_messages.append(msg)
        
        # Apply chat template
        text = GLOBAL_TOKENIZER.apply_chat_template(
            processed_messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # Tokenize
        inputs = GLOBAL_TOKENIZER(
            text,
            return_tensors='pt',
            truncation=True,
            max_length=GLOBAL_MAX_CONTEXT - 200  # Reserve space for output
        ).to(GLOBAL_MODEL.device)
        
        input_len = inputs['input_ids'].shape[1]
        
        # Calculate available tokens for generation
        max_new = min(
            request.max_tokens,
            GLOBAL_MAX_CONTEXT - input_len - 100
        )
        
        if max_new < 50:
            return JSONResponse(
                status_code=400,
                content={'error': f'Input too long ({input_len} tokens), no room for output'}
            )
        
        # Generate with KV cache (past_key_values=None starts fresh)
        with torch.no_grad():
            outputs = GLOBAL_MODEL.generate(
                **inputs,
                max_new_tokens=max_new,
                temperature=max(0.01, request.temperature),
                top_p=request.top_p,
                do_sample=True if request.temperature > 0 else False,
                pad_token_id=GLOBAL_TOKENIZER.pad_token_id,
                eos_token_id=GLOBAL_TOKENIZER.eos_token_id,
                use_cache=True,  # Enable KV cache
                return_dict_in_generate=True,
                output_scores=False
            )
        
        # Extract generated tokens (only new tokens)
        generated_ids = outputs.sequences[0][input_len:]
        response_text = GLOBAL_TOKENIZER.decode(
            generated_ids,
            skip_special_tokens=True
        )
        
        # Strip thinking tags if present (Qwen3 )
        if '' in response_text:
            # Remove everything between  and 
            import re
            response_text = re.sub(
                r'',
                '',
                response_text,
                flags=re.DOTALL
            ).strip()
        
        # Update stats
        stats['total_tokens'] += input_len + len(generated_ids)
        
        # Return OpenAI-compatible response
        return {
            'id': f'chatcmpl-{int(time.time() * 1000)}',
            'object': 'chat.completion',
            'created': int(time.time()),
            'model': 'qwen3',
            'choices': [{
                'index': 0,
                'message': {
                    'role': 'assistant',
                    'content': response_text
                },
                'finish_reason': 'stop'
            }],
            'usage': {
                'prompt_tokens': input_len,
                'completion_tokens': len(generated_ids),
                'total_tokens': input_len + len(generated_ids)
            }
        }
        
    except Exception as e:
        stats['errors'] += 1
        error_msg = f'{type(e).__name__}: {str(e)}'
        print(f'[ERROR] Chat completions: {error_msg}')
        traceback.print_exc()
        
        # Clear CUDA cache on OOM
        if 'out of memory' in str(e).lower():
            torch.cuda.empty_cache()
            print('[INFO] CUDA cache cleared')
        
        return JSONResponse(
            status_code=500,
            content={'error': error_msg}
        )

@app.get('/stats')
async def get_stats():
    """Get server statistics"""
    return {
        'total_requests': stats['total_requests'],
        'total_tokens': stats['total_tokens'],
        'errors': stats['errors'],
        'uptime_seconds': (datetime.now() - stats['start_time']).total_seconds()
    }

# Start server in background thread
def run_server():
    """Run FastAPI server in background"""
    uvicorn.run(
        app,
        host='0.0.0.0',
        port=8000,
        log_level='warning',
        access_log=False
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to be ready
print('Waiting for server to start...')
for i in range(30):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print(f'[OK] Server ready in {i+1} seconds')
            print(f'  Endpoint: http://localhost:8000')
            print(f'  Models: http://localhost:8000/v1/models')
            break
    except:
        pass
    time.sleep(1)
else:
    print('[ERROR] Server failed to start within 30 seconds')

print('=== SERVER READY ===')

In [ ]:
# ============================================================
# CELL 6: START JINA READER SERVER
# ============================================================
import threading
import time
import requests as req_lib
import trafilatura
from fastapi import FastAPI, Query
from fastapi.responses import JSONResponse
import uvicorn

print('=== STARTING JINA READER SERVER ===')

# Create Jina FastAPI app
jina_app = FastAPI(title='OMNIS Jina Reader', version='7.9')

@jina_app.get('/health')
async def jina_health():
    """Jina health check"""
    return {'status': 'ok', 'service': 'jina-reader'}

@jina_app.get('/extract')
async def extract(url: str = Query(..., description='URL to extract content from')):
    """Extract content from URL using trafilatura"""
    try:
        # Fetch URL content
        downloaded = trafilatura.fetch_url(
            url,
            no_ssl=False,
            timeout=20
        )
        
        if not downloaded:
            return JSONResponse(
                status_code=400,
                content={
                    'error': 'Failed to fetch URL',
                    'url': url
                }
            )
        
        # Extract text content
        text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=True,
            include_links=False,
            include_images=False,
            no_fallback=False,
            favor_precision=True
        )
        
        if not text or len(text.strip()) < 50:
            return JSONResponse(
                status_code=400,
                content={
                    'error': 'Content too short or extraction failed',
                    'url': url,
                    'content_length': len(text) if text else 0
                }
            )
        
        # Return extracted content
        return {
            'url': url,
            'content': text,
            'word_count': len(text.split()),
            'char_count': len(text),
            'status': 'success'
        }
        
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={
                'error': f'{type(e).__name__}: {str(e)}',
                'url': url
            }
        )

# Start Jina server in background
def run_jina():
    """Run Jina server in background"""
    uvicorn.run(
        jina_app,
        host='0.0.0.0',
        port=8001,
        log_level='warning',
        access_log=False
    )

jina_thread = threading.Thread(target=run_jina, daemon=True)
jina_thread.start()

# Wait for Jina server to be ready
print('Waiting for Jina server to start...')
for i in range(10):
    try:
        r = req_lib.get('http://localhost:8001/health', timeout=2)
        if r.status_code == 200:
            print(f'[OK] Jina server ready in {i+1} seconds')
            print(f'  Endpoint: http://localhost:8001')
            print(f'  Extract: http://localhost:8001/extract?url=...')
            break
    except:
        pass
    time.sleep(1)
else:
    print('[ERROR] Jina server failed to start')

print('=== JINA SERVER READY ===')

In [ ]:
# ============================================================
# CELL 7: CLOUDFLARE TUNNELS
# ============================================================
import subprocess
import re
import time

print('=== CREATING CLOUDFLARE TUNNELS ===')

def create_tunnel(port, service_name):
    """Create Cloudflare tunnel for given port"""
    print(f'Creating tunnel for {service_name} (port {port})...')
    
    process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{port}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    
    # Wait for tunnel URL to appear in stderr
    url = None
    for _ in range(30):
        line = process.stderr.readline()
        if not line:
            time.sleep(0.5)
            continue
        
        # Look for trycloudflare.com URL
        match = re.search(
            r'(https://[a-z0-9-]+\.trycloudflare\.com)',
            line
        )
        if match:
            url = match.group(1)
            break
    
    return process, url

# Create tunnels
llm_process, llm_url = create_tunnel(8000, 'LLM')
jina_process, jina_url = create_tunnel(8001, 'Jina')

# Display results
print()
print('=' * 70)
print('OMNIS-COURT COLAB READY! (v7.9 FINAL)')
print('=' * 70)

if llm_url and jina_url:
    print()
    print('TUNNEL URLs (copy these to config/platforms.json):')
    print()
    print(f'  LLM API:')
    print(f'    {llm_url}/v1')
    print()
    print(f'  Jina Reader:')
    print(f'    {jina_url}/extract')
    print()
    print('=' * 70)
    print('CONFIG UPDATE:')
    print()
    print('  "omnis_court": {')
    print(f'    "llm_api_url": "{llm_url}/v1",')
    print(f'    "jina_reader_url": "{jina_url}/extract",')
    print('    "llm_model_name": "qwen3",')
    print('    ...')
    print('  }')
    print()
    print('=' * 70)
    print('NEXT STEPS:')
    print('  1. Copy URLs above to config/platforms.json')
    print('  2. Deploy dashboard (Render/Streamlit)')
    print('  3. Test endpoints from your browser')
    print('  4. Close this tab (anti-idle is active)')
    print('=' * 70)
    
    # Store for later use
    TUNNEL_LLM_URL = llm_url
    TUNNEL_JINA_URL = jina_url
else:
    print('[ERROR] Failed to create tunnels')
    print(f'  LLM URL: {llm_url}')
    print(f'  Jina URL: {jina_url}')

In [ ]:
# ============================================================
# CELL 8: TEST ENDPOINTS
# ============================================================
import requests
import json

print('=== TESTING ENDPOINTS ===')
print()

# Test 1: LLM Health
print('[Test 1] LLM Health Check')
try:
    r = requests.get('http://localhost:8000/health', timeout=5)
    if r.status_code == 200:
        print('  [OK] LLM health check passed')
        print(f'  Response: {r.json()}')
    else:
        print(f'  [FAIL] Status {r.status_code}')
except Exception as e:
    print(f'  [FAIL] {e}')

print()

# Test 2: LLM Models
print('[Test 2] LLM Models List')
try:
    r = requests.get('http://localhost:8000/v1/models', timeout=5)
    if r.status_code == 200:
        print('  [OK] Models list retrieved')
        print(f'  Models: {r.json()["data"][0]["id"]}')
    else:
        print(f'  [FAIL] Status {r.status_code}')
except Exception as e:
    print(f'  [FAIL] {e}')

print()

# Test 3: LLM Chat Completion
print('[Test 3] LLM Chat Completion')
try:
    r = requests.post(
        'http://localhost:8000/v1/chat/completions',
        json={
            'model': 'qwen3',
            'messages': [
                {'role': 'user', 'content': 'Say "OK" in exactly 2 characters, nothing else.'}
            ],
            'max_tokens': 10,
            'temperature': 0.1
        },
        timeout=120
    )
    if r.status_code == 200:
        data = r.json()
        content = data['choices'][0]['message']['content']
        tokens = data['usage']['total_tokens']
        print('  [OK] Chat completion successful')
        print(f'  Response: {content[:50]}...')
        print(f'  Tokens: {tokens}')
    else:
        print(f'  [FAIL] Status {r.status_code}: {r.text[:200]}')
except Exception as e:
    print(f'  [FAIL] {e}')

print()

# Test 4: Jina Health
print('[Test 4] Jina Health Check')
try:
    r = requests.get('http://localhost:8001/health', timeout=5)
    if r.status_code == 200:
        print('  [OK] Jina health check passed')
    else:
        print(f'  [FAIL] Status {r.status_code}')
except Exception as e:
    print(f'  [FAIL] {e}')

print()

# Test 5: Jina Extract
print('[Test 5] Jina Content Extraction')
try:
    r = requests.get(
        'http://localhost:8001/extract',
        params={'url': 'https://en.wikipedia.org/wiki/Tennis'},
        timeout=30
    )
    if r.status_code == 200:
        data = r.json()
        print('  [OK] Content extraction successful')
        print(f'  Word count: {data.get("word_count", 0)}')
        print(f'  Char count: {data.get("char_count", 0)}')
    else:
        print(f'  [FAIL] Status {r.status_code}: {r.text[:200]}')
except Exception as e:
    print(f'  [FAIL] {e}')

print()

# Test 6: Tunnel URLs
print('[Test 6] Tunnel URL Verification')
try:
    if 'TUNNEL_LLM_URL' in globals():
        r = requests.get(f'{TUNNEL_LLM_URL}/v1/models', timeout=10)
        if r.status_code == 200:
            print('  [OK] LLM tunnel accessible')
        else:
            print(f'  [WARN] LLM tunnel status {r.status_code}')
    else:
        print('  [SKIP] No LLM tunnel URL')
except Exception as e:
    print(f'  [FAIL] {e}')

try:
    if 'TUNNEL_JINA_URL' in globals():
        r = requests.get(f'{TUNNEL_JINA_URL}/health', timeout=10)
        if r.status_code == 200:
            print('  [OK] Jina tunnel accessible')
        else:
            print(f'  [WARN] Jina tunnel status {r.status_code}')
    else:
        print('  [SKIP] No Jina tunnel URL')
except Exception as e:
    print(f'  [FAIL] {e}')

print()
print('=== ALL TESTS COMPLETED ===')